<h1>Chapter 2 - Large Language Models</h1>
<i>Exploring Large Language Model architecture</i>


<a href="..."><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="..."><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="..."><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](...)

---

This notebook is for Chapter 2 of the [An Illustrated Guide to AI Agents](...) book by [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/) and [Jay Alammar](https://www.linkedin.com/in/jalammar).

---

<a href="...">
<img src="https://learning.oreilly.com/covers/urn:orm:book:9798341662681/400w/" width="350"/></a>


### **[OPTIONAL]** - Installing Packages on <img src="https://colab.google/static/images/icons/colab.png" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** one of the following codeblock to install the dependencies for this chapter. If you want to use a cloud provider, you only need to run the following code block:

In [ ]:
# %%capture
# !pip install illustrated-agents

---

💡 **NOTE**: If you want to use the GPU with `ollama`, then you will have to select a GPU first. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**. 

Then, **uncomment** and run this codeblock:

---

In [ ]:
# !apt-get install -y zstd > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh
# !nohup ollama serve > /dev/null 2>&1 & sleep 3 && ollama pull gemma3:12b &

# Adding the `"Brain"`

Throughout this chapter, we covered the foundational structure of common LLMs, namely the Transformer. In this accompanying notebook, we will be adding *any* LLM to our `TinyAgent` as our first step towards autonomy.

![../images/ch2.png](../images/ch2.png)

To do so, we will have to consider which LLM we want to be using. This decision can be quite complex as it relates to your use case, your hardware (requirements), etc. Instead of providing you a full list of (quickly outdated) LLMs, we instead decided to allow readers to run *any* LLM with [`LiteLLM`](). `LiteLLM` is essentially a wrapper around common LLM inference engines that attempts to standardize how they can be used to the OpenAI format. Instead of having to learn specific APIs for [`vLLM`](https://github.com/vllm-project/vllm) vs. [`ollama`](https://github.com/ollama/ollama), `LiteLLM` makes it all the same to use. 

More specifically, many LLM inference engines create servers that expose a REST API that you can use for inference. This is especially handy when you have a dedicated machine to run models on and want to use another device to access. This is not limited to cloud solutions, we purposefully chose `LiteLLM` since it also allows you to run your models locally. To serve as many people reading this, we will be making examples of various LLMs through various methods (local vs. cloud).

Let us first explore various LLM inference engines for running your LLM. 

# **Inference Engines**

We split the inference engines by how they are typically used, namely either for local inference on your device on or using the cloud (which often is a propriety model).

## **Local**

As big fans of open-weight LLMs, we want users to be able to create and run Agents on their local devices. This setup can be a bit tricky since it requires setting up your environment, CUDA, Transformers, etc. Fortunately, there are a couple of packages that make this much easier.

### **Ollama**

[`Ollama`](https://ollama.com/) is arguably one of the most straightforward methods to run LLMs locally without needing to know much about how to setup your environment or offload layers to your GPU, Ollama does all of that for you and more. You can [download](https://ollama.com/) Ollama from their website. After installation, you can [download any model](https://ollama.com/search) using the following command

```bash
# Download Gemma 3 to be used locally
ollama pull gemma3:12b
```

After downloading the model, you can use the interface of the software to talk to your LLM:

![../images/ollama.png](../images/ollama.png)

As most local inference engines, it exposes a server with a REST API that we can query to run inference. `LiteLLM` takes care of that for us, so all you have to do is download `ollama` and a model that you want to run (we advise `gemma3:12b` but more on that later!).

### **LM Studio**

Another popular method is using [`LM Studio`](https://lmstudio.ai/). Like Ollama, `LM Studio` can be downloaded and used to run models locally without having in-depth knowledge about setting up hardware-specific inference. After installing the software you first need to click on the "search" icon so we can begin downloading the model:

![../images/lmstudio1.png](../images/lmstudio1.png)

Then, search for your model (we are using gemma 3 with 12 billion parameters) and download like so:

![../images/lmstudio2.png](../images/lmstudio2.png)

Like Ollama, we can use the server endpoint to run our model. To check which adress it is using:

![../images/lmstudio3.png](../images/lmstudio3.png)

### **Llama.cpp**

If you have experience with the terminal, Linux, setting up servers, etc. and want to have more control, then we advise the popular [`llama.cpp`](https://github.com/ggml-org/llama.cpp) package. This inference engines runs fully locally but requires you to install it yourself. Fortunately, [installation](https://github.com/ggml-org/llama.cpp/blob/master/docs/install.md) only requires a single line of code:

```sh
# Winget (Windows)
winget install llama.cpp

# Homebrew (Mac and Linux)
brew install llama.cpp
```

This will install `llama.cpp` on your device along with the `llama-server`. The latter is what `LiteLLM` will use to query your model. Before doing so, we first need to download the quantized model. Quantization is essentially a compression of a model's parameters in lower precision, which allows for much smaller models without too much of a drop in performance.

As always, we recommend downloading a [4-bit quantized Gemma 3](https://huggingface.co/unsloth/gemma-3-12b-it-GGUF/blob/main/gemma-3-12b-it-Q4_K_M.gguf) model.

After doing so, you can start your server up with:

```bash
llama-server -m .\gemma-3-12b-it-Q4_K_M.gguf --port 8080
```

The server and UI can then be accessed through your browser: http://localhost:8080

![../images/llamacpp.png](../images/llamacpp.png)


### **Llama-cpp-python**

There exists a python wrapper around `llama.cpp` which makes it easier to run 

If you have experience with the terminal, Linux, setting up servers, etc. and want to have more control, then we advise the popular [`llama.cpp`](https://github.com/ggml-org/llama.cpp) package. This inference engines runs fully locally but requires you to install it yourself. Fortunately, [installation](https://github.com/ggml-org/llama.cpp/blob/master/docs/install.md) only requires a single line of code:

```sh
# Winget (Windows)
winget install llama.cpp

# Homebrew (Mac and Linux)
brew install llama.cpp
```

This will install `llama.cpp` on your device along with the `llama-server`. The latter is what `LiteLLM` will use to query your model. Before doing so, we first need to download the quantized model. Quantization is essentially a compression of a model's parameters in lower precision, which allows for much smaller models without too much of a drop in performance.

As always, we recommend downloading a [4-bit quantized Gemma 3](https://huggingface.co/unsloth/gemma-3-12b-it-GGUF/blob/main/gemma-3-12b-it-Q4_K_M.gguf) model.

After doing so, you can start your server up with:

```bash
llama-server -m .\gemma-3-12b-it-Q4_K_M.gguf --port 8080
```

The server and UI can then be accessed through your browser: http://localhost:8080

![../images/llamacpp.png](../images/llamacpp.png)

If you want to use `llama-cpp-python` in Google Colab, please run the following:

```bash
!pip install illustrated-agents[cuda]
!wget https://huggingface.co/unsloth/gemma-3-12b-it-GGUF/resolve/main/gemma-3-12b-it-Q4_K_M.gguf
!nohup python3 -m llama_cpp.server --model gemma-3-12b-it-Q4_K_M.gguf --n_gpu_layers -1 &
```

## **Cloud**

Although it is nice to run things locally, not everyone has access to powerful hardware capable enough to run these models. Instead, we can use cloud providers and use their (**free!**) tiers to run the Agent(s) throughout this book.

### **Google**

Google has some amazing models to use (like the Gemini family of models) but also allow you to run their open-weight models (such as Gemma 3), which makes this a very interesting platform to use. We will be using the **Free Tier** which allows us to use some of these models with rate limits. To make sure we do not hit the rate limits, we advise to use their "flash" variants or their open-weight models. 

First, go to the [Gemini documentation](https://ai.google.dev/gemini-api/docs/api-key) and click on "Create or view a Gemini API Key":


![../images/google1.png](../images/google1.png)

Then, click on "Create API Key":

![../images/google2.png](../images/google2.png)

Before you can actually create a key, you may need to first create a project if you hadn't already done so. Then select it and create the key:

![../images/google3.png](../images/google3.png)

Finally, copy your API key to use throughout this book.

### **OpenAI**

You can also use one of OpenAI's models, which do not have a Free tier unfortunately. That said, if you already happen to have an API Key, you can find it [here](https://platform.openai.com/api-keys) and simply use that for one of their models.

### **Claude**

Although we haven't tried it ourselves yet, free credits are given to new users to test the API. We are not sure whether this is sufficient for testing the entire book, so be aware. You can find more about their pricing [here](https://platform.claude.com/docs/en/about-claude/pricing).





# **Which model should I use?**

Here it is, a question that it is incredibly difficult to keep current. Fortunately, there are many extremely capable small and large models that we can use these days. Throughout this book, we have been testing all code primarily with the smallest, most capable, and easiest to use model that we could find, namely [`Gemma 3`](https://deepmind.google/models/gemma/gemma-3/). It comes in several sizes and we have been succesfully testing the model with 12 billion parameters at a quantization (compression) of 4 bits. This model can be run locally if you have 8GB of VRAM available (although 12GB is preferred). That said, if you do not mind waiting a bit, this model also runs quite well locally. Although there are other amazing local models, they often require specific templates (like `GPT-OSS`). Although we have not tested them, these are models that are at the time of writing (Januari 2026) considered one of the best in their size category:

* Qwen 3
* GPT-OSS
* Gemma 3
* GLM-4.7 Flash
* Phi-4

Larger, open-weight models include:

* Kimi-K2
* DeepSeek V3.2

With respect to cloud providers we highly recommend using Google's offering since they not only provide a free tier, but also endpoints to their open-weight models.

# **The LLM Wrapper**

Now that we have explored all various types of models and inference backends, let's create our `LLM` class that we use throughout this book:

Before doing that, let's explore this `"Brain"` that we want to add to our `TinyAgent`:

In [62]:
from litellm import completion


class LLM:
    def __init__(self, model: str, **kwargs):
        """Initialize the LLM with the given model."""
        self.model = model
        self.kwargs = kwargs

    def generate(self, messages: list[dict]) -> str:
        """Generate a response from the LLM given a list of messages."""
        response = completion(model=self.model, messages=messages, **self.kwargs)
        return response


The structure of our LLM is rather straightforward, we create the `LLM` class and use the `litellm.completion` to run our model. Note how it gives back a `response`. In this `response` you can find not only the output of the model but also its intermediate reasoning (if any) and other metadata. To use it, use one of the following models that you have prepared:

In [63]:
import os

# Ollama
llm = LLM(model="ollama/gemma3:12b")

# Llama.cpp server
# llm = LLM(model="openai/gemma-3-12b-it-Q4_K_M", api_base="http://localhost:8080", api_key="sk-no-key-required")

# Llama-cpp-python server
# llm = LLM(model="openai/gemma-3-12b-it-Q4_K_M.gguf", api_base="http://localhost:8000/v1/", api_key="sk-no-key-required")

# LM Studio
# llm = LLM(model="lm_studio/gemma-3-12b-it", api_base="http://localhost:1234/v1", api_key="sk-no-key-required")

# Google's Gemini / Gemma
# os.environ['GEMINI_API_KEY'] = "YOUR_GEMINI_API_KEY"
# llm = LLM(model="gemini/gemini-2.5-flash")
# llm = LLM(model="gemini/gemma-3-12b-it")

We can then use the `messages` structure to talk to the LLM:

In [64]:
response = llm.generate([{"role": "user", "content": "What is 1+1?."}])
response

ModelResponse(id='chatcmpl-9e9c954f-f80e-4f10-9130-655a4e563d1b', created=1769424461, model='ollama/gemma3:12b', object='chat.completion', system_fingerprint=None, choices=[Choices(finish_reason='stop', index=0, message=Message(content='1 + 1 = 2\n', role='assistant', tool_calls=None, function_call=None, provider_specific_fields=None, reasoning_content=None))], usage=Usage(completion_tokens=9, prompt_tokens=21, total_tokens=30, completion_tokens_details=None, prompt_tokens_details=None))

Note that the `ModelResponse` has a two fields that are of interest to us:

* `choices` -- The output of the LLM
* `usage` -- How many tokens were used

Looking at `choices` gives us back a list of answers, but this is typically just a single response since the model is asked to only generate a single answer. When we look what is in this `response.choices[0]` we can see that it returns a `Message` with the answer (`content`) but also whether there was any explicit reasoning or tool calls. For the purpose of this book, we are not going to be using the tool and reasoning fields as this is behavior we want to create ourselves and not every model supports this.

In [66]:
response.choices[0]

Choices(finish_reason='stop', index=0, message=Message(content='1 + 1 = 2\n', role='assistant', tool_calls=None, function_call=None, provider_specific_fields=None, reasoning_content=None))

Since we are only interested in the answer (`content`) of the model, we need to adjust our `LLM` to return only that:

In [1]:
class LLM:
    def __init__(self, model: str, **kwargs):
        """Initialize the LLM with the given model."""
        self.model = model
        self.kwargs = kwargs

    def generate(self, messages: list[dict]) -> str:
        """Generate a response from the LLM given a list of messages."""
        # Call the chat completion function
        response = completion(model=self.model, messages=messages, **self.kwargs)

        # Extract and return the output text
        output = response.choices[0].message.content
        return output


Let's explore these steps in more detail:

In [12]:
from illustrated_agents.chapters.ch2 import llm_annotated; llm_annotated

Finally, we can call the LLM with our updated class and extract the response only:

In [69]:
# Re-initialize the LLM with the updated class
llm = LLM(model="ollama/gemma3:12b")

# Generate a response
response = llm.generate([{"role": "user", "content": "What is 1+1?."}])
print(response)

'1 + 1 = 2\n'

# Updating `agent.py`

We update our `TinyAgent` to have a brain and actually be able to answer:

In [ ]:
class TinyAgent:
    """A minimal, modular, and educational agent framework."""

    def __init__(self, llm: LLM):
        self.llm = llm
        self.memory = None  # Chapter 4: Add Memory
        self.tools = None  # Chapter 5: Add Tools
        self.planner = None  # Chapter 6: Add Planning
        self.reflector = None  # Chapter 6: Add Reflection

    def run(self, task: str) -> str:
        """Run the agent on a task."""
        return self._step(task)

    def _step(self, task: str) -> str:
        """Perform a single step."""
        messages = [{"role": "user", "content": task}]
        return self.llm.generate(messages)

Here is a nicer overview of the changes that we made to `agent.py` (red is removed and green is added code):

In [2]:
from illustrated_agents.chapters.ch2 import tinyagents_diff; tinyagents_diff

We can then run our agent as follows:

In [74]:
agent = TinyAgent(llm=llm)
response = agent.run("What is 2 + 2?")
print(response)

2 + 2 = 4



Our "Agent" is still nothing more than the LLM and has no additional behavior/capabilities that we can showcase yet. For that, we need to add more modules, such as **memory**, **tools**, and **planning**.